# Precision, Recall, and Threshold - Applied with SVM

In this worksheet you will apply an SVM-based precision-recall analysis to the [Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), a built-in sklearn dataset. You will binarize the multi-class target to detect a specific digit, tune an SVM pipeline, and explore the threshold-precision-recall curve.

***Summary***
1. [Load and Explore the Data](#load-data)
2. [Data Preparation](#data-prep)
3. [SVM Pipeline and Hyperparameter Tuning](#svm-pipeline)
4. [Threshold Analysis and Model Evaluation](#threshold-eval)
5. [Apply to a New Domain](#new-domain)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

***
<a id='load-data'></a>
## 1. Load and Explore the Data

The Digits dataset contains 8x8 grayscale images of handwritten digits (0-9). Each image is flattened to a 64-feature vector. We binarize the target: detecting the digit '3' (positive class, label = 1) vs. all other digits (negative class, label = 0). Detecting a specific digit among many is a realistic use case in OCR (optical character recognition) systems.

**Q1a) Load the Digits dataset. Store the feature matrix in `X` and binarize the target: set `y` to 1 where the original label equals 3, else 0. Print the shape of `X` and the class counts.**

*Hint:* Use [`sklearn.datasets.load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

In [ ]:
from sklearn.datasets import load_digits

### YOUR CODE HERE ###
data = load_digits()
X = ...
y = ...

#print('X shape:', X.shape)
#print('Positive class (digit 3) count:', (y == 1).sum())
#print('Negative class count:', (y == 0).sum())

**Q1b) Visualize one image from the positive class (digit 3) and one from the negative class. Use `data.images` to get the 2D image arrays and `plt.imshow` to display them side by side.**

Expected output: a matplotlib figure showing two 8x8 digit images side by side.

`# TODO: capture expected plot output as assets/q1b_expected.PNG`

In [ ]:
# Find indices for one positive and one negative example
idx_positive = np.where(y == 1)[0][0]  # don't change these lines
idx_negative = np.where(y == 0)[0][0]  # don't change these lines

### YOUR CODE HERE ###
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
# axes[0]: show image at idx_positive with title 'Digit 3 (positive)'
# axes[1]: show image at idx_negative with title label
...
plt.tight_layout()
plt.show()

***
<a id='data-prep'></a>
## 2. Data Preparation

Split the data into train and test sets before any fitting. Use the same three-way split strategy from the lesson notebooks.

**Q2) Split the data into a test set (15%) and a training set using `random_state=0`. Then split the training set into a tuning-train set and validation set (15% of training). Store the four arrays as `X_train`, `X_test`, `y_train`, `y_test`, `X_t`, `X_v`, `y_t`, `y_v`.**

In [ ]:
random_state = 0   # don't change
test_size = 0.15   # don't change

### YOUR CODE HERE ###
X_train, X_test, y_train, y_test = ...
X_t, X_v, y_t, y_v = ...

#print('Train+Val size:', len(X_train), '| Test size:', len(X_test))
#print('Tuning train:', len(X_t), '| Validation:', len(X_v))

***
<a id='svm-pipeline'></a>
## 3. SVM Pipeline and Hyperparameter Tuning

Build a Pipeline with StandardScaler and SVC, then tune the regularization parameter C over a logarithmic range.

**Q3a) Build a Pipeline with `StandardScaler` and `SVC(kernel='rbf', gamma='auto', probability=True, max_iter=int(1e6))`. Store it in `pipeline`.**

*Hint:* Use [`sklearn.pipeline.Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

In [ ]:
### YOUR CODE HERE ###
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', ...)
])

**Q3b) Tune the `C` parameter over `np.logspace(0, 3, 8)` using 4-fold `StratifiedKFold` and `scoring='recall'`. Fit the grid search on `X_t, y_t`. Store the best estimator in `pipeline_best_recall`.**

In [ ]:
split = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)  # don't change
parameters = {'svc__C': np.logspace(0, 3, 8)}                        # don't change

### YOUR CODE HERE ###
grid_recall = ...
...
pipeline_best_recall = ...

#print('Best C (recall-optimized):', pipeline_best_recall.named_steps['svc'].C)

**Q3c) Compute and print the cross-validation recall and precision for `pipeline_best_recall` on the tuning training set `X_t, y_t`. Store them in `cv_recall` and `cv_precision`.**

In [ ]:
### YOUR CODE HERE ###
cv_recall = ...
cv_precision = ...

#print(f'CV Recall:    {cv_recall:.3f}')
#print(f'CV Precision: {cv_precision:.3f}')

***
<a id='threshold-eval'></a>
## 4. Threshold Analysis and Model Evaluation

Sweep the decision threshold on the validation set, then evaluate the final model on the test set.

**Q4a) Fit `pipeline_best_recall` on `X_t`. Sweep 20 threshold values between 0.05 and 0.95. At each threshold, compute precision and recall on the validation set `X_v, y_v`. Store results in `precisions` and `recalls`. Plot both curves.**

Expected output: a matplotlib figure with precision (red) and recall (blue) vs. threshold.

`# TODO: capture expected plot output as assets/q4a_expected.PNG`

In [ ]:
pipeline_best_recall.fit(X_t, y_t)

n = 20
thresh = np.linspace(0.05, 0.95, n)
precisions = np.zeros(n)
recalls = np.zeros(n)

### YOUR CODE HERE ###
for i, t in enumerate(thresh):
    ...

fig, ax = plt.subplots(figsize=(8, 4))
### YOUR CODE HERE ###
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall vs. Threshold (Digit 3 Detection, SVM)')
ax.legend()
plt.tight_layout()
plt.show()

**Q4b) From the threshold sweep, identify the threshold that maximizes recall while keeping precision above 0.80. If no threshold satisfies both, use the threshold with the highest recall. Store the chosen threshold in `thresh_chosen`.**

In [ ]:
### YOUR CODE HERE ###
thresh_chosen = ...

#print(f'Chosen threshold: {thresh_chosen:.2f}')

**Q4c) Retrain `pipeline_best_recall` on the full `X_train, y_train` set. Evaluate on `X_test` at `thresh_chosen`. Compute and print test recall and precision. Store them in `test_recall` and `test_precision`.**

Expected output: confusion matrix and two metric values printed to the console.

In [ ]:
pipeline_best_recall.fit(X_train, y_train)

### YOUR CODE HERE ###
y_pred_test = ...
C_test = confusion_matrix(y_test, y_pred_test)
tp, fp, fn = C_test[1, 1], C_test[0, 1], C_test[1, 0]

test_recall = ...
test_precision = ...

#print('Test confusion matrix:')
#print(C_test)
#print(f'\nTest recall:    {test_recall:.3f}')
#print(f'Test precision: {test_precision:.3f}')

***
<a id='new-domain'></a>
## 5. Apply to a New Domain

Apply the SVM precision-recall framework to a different task: detecting wine class 0 (a high-quality wine type) from the Wine Recognition dataset. This dataset has 13 chemical features and 178 samples, a different scale from the 64-feature Digits dataset.

**Q5a) Load the Wine Recognition dataset using `load_wine`. Binarize to detect class 0 (positive, label = 1) vs. classes 1 and 2 (negative, label = 0). Split into 80/20 train/test with `random_state=0`. Store in `X_wine_train`, `X_wine_test`, `y_wine_train`, `y_wine_test`.**

*Hint:* Use [`sklearn.datasets.load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)

In [ ]:
from sklearn.datasets import load_wine

### YOUR CODE HERE ###
data_wine = load_wine()
X_wine = ...
y_wine = ...  # 1 = class 0 (positive), 0 = classes 1 and 2 (negative)

X_wine_train, X_wine_test, y_wine_train, y_wine_test = ...

#print('X_wine shape:', X_wine.shape)
#print('Class 0 (positive) count:', (y_wine == 1).sum())

**Q5b) Build the same SVM pipeline and run a GridSearchCV with `scoring='recall'` over `np.logspace(0, 3, 8)` using 4-fold stratified CV. Fit on `X_wine_train`. Store the best estimator in `pipeline_wine_recall`. Evaluate on `X_wine_test` at threshold 0.5 and print test recall and precision. Store them in `wine_recall` and `wine_precision`.**

In [ ]:
### YOUR CODE HERE ###
pipeline_wine = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', gamma='auto', probability=True, max_iter=int(1e6)))
])

pipeline_wine_recall = ...
pipeline_wine_recall.fit(X_wine_train, y_wine_train)

y_pred_wine = (pipeline_wine_recall.predict_proba(X_wine_test)[:, 1] >= 0.5).astype(int)
C_wine = confusion_matrix(y_wine_test, y_pred_wine)
tp_w, fp_w, fn_w = C_wine[1, 1], C_wine[0, 1], C_wine[1, 0]

wine_recall = ...
wine_precision = ...

#print(f'Wine test recall:    {wine_recall:.3f}')
#print(f'Wine test precision: {wine_precision:.3f}')

*ANSWER HERE*

Compare the recall achieved on digit-3 detection (`test_recall`) versus wine class-0 detection (`wine_recall`). Which classification problem is harder for the SVM? Propose one structural reason why based on the feature counts and sample sizes of the two datasets.